In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance


panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data_no_dupes.csv"
panel_data = pd.read_csv(panel_data_path)

In [ ]:
panel_data['Date'] = pd.to_datetime(panel_data['Date'])
panel_data = panel_data.sort_values('Date', ascending= True)
panel_data['CovidAffectedPrediction'] = (
    panel_data['Year'].isin([2019, 2020, 2021])
).astype(int)
panel_data = panel_data.drop(columns = 'Date')

### Train/validate/test split
Train/validate/test split must be time-aware. Entries in the train set should predate entries in the validation set which should predate entries in the test set. The dataframe year_counts is used to decide which years should be used for the train/validation/test sets.

In [ ]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

- 2022 and before used as train set
- 2023 used for validation set
- 2024 used for test set

In [ ]:
panel = panel_data.drop(columns = ['Name', 'WeightClassKg', 'Sex'])
train = panel[panel['Year']<=2022].copy()
validate = panel[panel['Year']==2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'CompetesNextYear')
train_y = train['CompetesNextYear']
validate_X = validate.drop(columns = 'CompetesNextYear')
validate_y = validate['CompetesNextYear']
test_X = test.drop(columns = 'CompetesNextYear')
test_y = test['CompetesNextYear']

### Establish baseline accuracy

In [ ]:
clf_baseline = DummyClassifier(strategy="most_frequent")
clf_baseline.fit(train_X, train_y)
baseline_accuracy = clf_baseline.score(test_X, test_y)
print(baseline_accuracy)

### Initial coarse feature selection using random forest feature importances

In [ ]:
clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf.fit(train_X, train_y)

#will be using performance on validation set for feature selection
def evaluate_model(classifier, X_val, y_val):
    preds = classifier.predict(X_val)
    return {
        "f1":        f1_score(y_val, preds),
        "recall":    recall_score(y_val, preds),
        "accuracy":  accuracy_score(y_val, preds),
        "precision": precision_score(y_val, preds) 
    }

#performance of classifier in validation set when trained on all features. will be used for comparison.
all_feats_metrics = evaluate_model(clf, validate_X, validate_y)

In [ ]:
all_feats_metrics

Approach 1: permutation importance and then drop features with negative importance. 

<b>Approach 2 </b>: use random forest for feature importance.

In [ ]:
#using random forest to get feature importances 
clf_feat_selection = RandomForestClassifier(random_state=42)
clf_feat_selection.fit(train_X, train_y)
importances = clf_feat_selection.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)
feature_importance_df

In [ ]:
#getting rid of features of order of magnitude e-03 or lower and traing HistGradientBoostingClassifier
important_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.01]
features = important_feats['Feature'].to_list()
train_rf_X= train_X[features]
validate_rf_X = validate_X[features]

clf_rf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_rf.fit(train_rf_X, train_y)

rf_metrics = evaluate_model(clf_rf, validate_rf_X, validate_y)

In [ ]:
result = permutation_importance(clf, validate_X, validate_y, n_repeats=10, random_state=42)

feature_importance_perm = pd.DataFrame({
    'Feature': validate_X.columns,
    'Importance': result.importances_mean
}).sort_values('Importance', ascending=False, ignore_index=True)

perm_feats = feature_importance_perm.loc[feature_importance_perm['Importance']>0, 'Feature'].to_list()
train_perm_X = train_X[perm_feats]
validate_perm_X = validate_X[perm_feats]
clf_perm_feats = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_perm_feats.fit(train_perm_X, train_y)

perm_metrics = evaluate_model(clf_perm_feats, validate_perm_X, validate_y)

In [ ]:
print(all_feats_metrics)
print(rf_metrics)
print(perm_metrics)

Will do a coarse feature reduction based on random forest feature ranking before training HistBoostClassifier, as this shows no meaningful change from using all features. Whereas using permutation importance shows significant decrease in f1 and recall on validation set.

In [ ]:
train_perc_X = train_rf_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])

validate_perc_X = validate_rf_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])


clf_perc = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_perc.fit(train_perc_X, train_y)

In [ ]:
perc_metrics = evaluate_model(clf_perc, validate_perc_X, validate_y)
print(rf_metrics)
print(perc_metrics)

Prioritising recall (not missing churners) on the assumption that the cost of a retention intervention is lower than the expected value of a lost customer. However, this should be verified. Therefore keeping the non-percentage based improvement features, as recall decreases when they are removed.

### How many features to keep
Will reevaluate feature importance using random forest, in case dropping features has changed importance ranking.
Will take top N features where N is determined by taking top n features and seeing which value of n yields best performance on validation set. Can then use top N features to retrain classifier and evaluate performance using test set.


In [ ]:
clf_ablation = RandomForestClassifier(random_state=42)
clf_ablation.fit(train_rf_X, train_y)
importances_ablation = clf_ablation.feature_importances_

In [ ]:
ablation_df = pd.DataFrame({
    'Feature': train_rf_X.columns,
    'Importance': importances_ablation
}).sort_values('Importance', ascending=False, ignore_index =True)
ablation_df

# reduced_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.01]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0).reset_index(drop = True)

In [ ]:
from tqdm import tqdm

In [ ]:
results = []

for i in tqdm(range(len(ablation_df))):
    features = ablation_df.loc[:i, 'Feature'].to_list()

    train_n_X = train_rf_X[features]
    validate_n_X = validate_rf_X[features]

    clf_n = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
    clf_n.fit(train_n_X, train_y)
    
    preds_n = clf_n.predict(validate_n_X)
    acc_n = accuracy_score(validate_y, preds_n)
    f1_n = f1_score(validate_y, preds_n)
    precision_n = precision_score(validate_y, preds_n)
    recall_n = recall_score(validate_y, preds_n)

    results.append({'features': features, 
                    'accuracy': acc_n,'f1': f1_n, 'precision': precision_n, 'recall':recall_n})

results_df = pd.DataFrame(results)

In [ ]:
results_df

In [ ]:

results_df['feature_added'] = results_copy['features'].apply(lambda x: x[-1])
results_df

In [ ]:
feats_list = results_df.loc[15]['features']

train_f_X = train_rf_X[feats_list]
validate_f_X = validate_rf_X[feats_list]


clf_f = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)

clf_f.fit(train_f_X, train_y)

f_metrics = evaluate_model(clf_f, validate_f_X, validate_y)

f_metrics

In [ ]:
feats_list

Early stopping was tried but model performed better without so will tune max_iter instead 

In [ ]:
feats_list = results_df.loc[15]['features']

train_f_X = train_rf_X[feats_list]
validate_f_X = validate_rf_X[feats_list]


clf_f = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=500,
    max_depth=6,
    random_state=42
)

clf_f.fit(train_f_X, train_y)

f_metrics = evaluate_model(clf_f, validate_f_X, validate_y)

f_metrics

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import pandas as pd

full_train_X = pd.concat([train_f_X, validate_f_X])
full_train_y = pd.concat([train_y, validate_y])
full_train_X['Year'].is_monotonic_decreasing

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_iter': [200, 500, 1000],
    'max_depth': [3, 6, 9]
}

grid_search_f1 = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=tscv,
    n_jobs=-1,
    verbose=1
)
grid_search_f1.fit(full_train_X, full_train_y)

grid_search_recall = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_grid,
    scoring='recall',
    cv=tscv,
    n_jobs=-1,
    verbose=1
)
grid_search_recall.fit(full_train_X, full_train_y)

print(f"Best params (F1):     {grid_search_f1.best_params_}")
print(f"Best params (Recall): {grid_search_recall.best_params_}")